# GeoSuite Nickel — notebook pendamping workshop
**Ghozian Karami / Orebit.id • 4–5 November 2026 • Data sintetis340lubang**

Gunakan ini untuk memahami perjalanan **Core → Assay → Resource** dengan empat CSV yang sama seperti demonstrasi GeoSuite. Jalur utama tetap import/demonstrasi aplikasi. Notebook adalah jalur Python pendamping, bukan tampilan aplikasi lain.

**Baca hasil di GitHub. Jalankan di Jupyter atau Colab** setelah mengunduh notebook. GitHub tidak menjalankan kernel. Pilih **Run All** dari lingkungan baru; ubah `MODE` menjadi `loft` untuk interpretasi alternatif dan jalankan ulang semua sel. Jangan melewati sel yang gagal.

### Sebelum mulai
- Pengukuran tidak diubah; missing bukan nol. CRS **EPSG:32751**, koordinat/depth/elevasi meter, Ni persen.
- Ini data sintetis, bukan endapan nyata. Lisensi CC BY4.0, kredit Orebit.id/Ghozian Karami.
- **Interpretasi solid, geometri komposit presisi penuh dan clipped block support diberikan sebagai input geologi referensi dari GeoSuite.** Python menghitung clipping raw interval, komposit, pasangan variogram, NN/IDW/OK, tonase dan kurva dari input tersebut. Notebook tidak menggantikan digitize section, membuktikan geologi independen, atau membuat resource patuh SNI/KCMI/JORC.
- Survey contoh seluruhnya vertikal. Rumus koordinat vertikal di sini menolak data miring; gunakan desurvey minimum-curvature GeoSuite untuk data miring.
- Sel kode telah diperiksa dengan runner Python berurutan; verifikasi kernel Jupyter dicatat terpisah pada laporan distribusi. Jangan menyebut pemeriksaan Python sebagai uji peserta/perangkat.

| Materi workshop | Tahap notebook | Padanan GeoSuite |
|---|---|---|
| Integrasi data bor & validasi |1–2|Core Import/Validation/Desurvey/log|
| EDA & geological domaining |3–5|Assay Stats → Core Domain|
| Compositing |6|Assay Composite|
| Variografi |7|Resource Variography|
| Estimasi & block model |8–9|Resource Block Model/Estimation|
| Evaluasi & grade–tonnage |10–11|Resource CV/Grade–Tonnage/Report|


## 0 · Siapkan berkas
Ekstrak paketv2 dan simpan notebook di dekat foldernya. Jika dataset belum ada, sel berikut mengambil paket publik yang **dipatok commit dan SHA-256**, lalu memverifikasinya. Tidak memakai akses privat Codex atau orebit-ops.

[Unduh paket peserta](https://raw.githubusercontent.com/ghoziankarami/orebit-datasets/863ea4f10f0eff9cc4bc43d5eb85bd28d50900a3/workshops/2026-11-05/GeoSuite_Workshop_Nickel_v2.zip) · [Sumber data asli](https://github.com/ghoziankarami/orebit-datasets/tree/6b93ac48df79264fda20e4ffc10716280bd2ee8a/02-nikel-laterit)

Dependensi yang dipakai saat pemeriksaan: Python3.12, NumPy2.3.5, SciPy1.17.0, pandas2.2.3, matplotlib3.10.8. `requirements.txt` disediakan untuk environment terpisah. Aplikasi GeoSuite sendiri tidak memerlukan Python.


In [1]:
from pathlib import Path
import hashlib
import json
import urllib.request
import zipfile
import os

PACKAGE_URL = "https://raw.githubusercontent.com/ghoziankarami/orebit-datasets/863ea4f10f0eff9cc4bc43d5eb85bd28d50900a3/workshops/2026-11-05/GeoSuite_Workshop_Nickel_v2.zip"
PACKAGE_SHA256 = "5707fba03afc9744b229de960e183f131b814de3e1f6dca2156af17b98f04d0c"
candidates = [
    Path.cwd(),
    Path.cwd() / "GeoSuite_Workshop_Nickel_v2",
    Path.cwd().parent / "GeoSuite_Workshop_Nickel_v2",
]
root = next((p for p in candidates if (p / "dataset/collar.csv").is_file()), None)
if root is None:
    archive = Path.cwd() / "GeoSuite_Workshop_Nickel_v2.zip"
    with urllib.request.urlopen(PACKAGE_URL, timeout=60) as response:
        payload = response.read()
    assert (
        hashlib.sha256(payload).hexdigest() == PACKAGE_SHA256
    ), "Package checksum mismatch: stop"
    archive.write_bytes(payload)
    with zipfile.ZipFile(archive) as bundle:
        for name in bundle.namelist():
            assert (
                (Path.cwd() / name).resolve().is_relative_to(Path.cwd().resolve())
            ), "Unsafe archive path"
        bundle.extractall(Path.cwd())
    root = Path.cwd() / "GeoSuite_Workshop_Nickel_v2"
os.chdir(root)
print("Dataset tersedia. Empat CSV diperiksa pada sel berikutnya.")

Dataset tersedia. Empat CSV diperiksa pada sel berikutnya.


## 1 · Import empat tabel yang sama
**Tindakan:** jalankan sel, lalu bandingkan nama kolom dengan mapping Core. `MODE='pair'` memakai SAP dari dua surface; `MODE='loft'` memakai interpretasi tiga section. Kedua footprint berbeda.

**Cek:**340collar/680survey/8.751assay/1.344lithology. Jangan memakai tombol Sample bawaan untuk memperoleh angka ini: sample Core adalah latihan QC yang berbeda. Tidak ada pemilihan kadar atau pengubahan nilai pada340lubang ini.


In [2]:
from pathlib import Path
import hashlib
import os
import numpy as np
import pandas as pd
from scipy.linalg import solve
from scipy.spatial.distance import cdist, pdist
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / "dataset"
REF = ROOT / "output"
RESULTS = ROOT / "hasil-python"
RESULTS.mkdir(exist_ok=True)
MODE = "pair"  # 'pair' or 'loft'; rerun all cells after changing this interpretation.
assert MODE in ("pair", "loft")
raw = {
    name: pd.read_csv(DATA / f"{name}.csv")
    for name in ("collar", "survey", "assay", "litho")
}
expected_counts = {"collar": 340, "survey": 680, "assay": 8751, "litho": 1344}
assert {name: len(df) for name, df in raw.items()} == expected_counts
hashes = json.loads((REF / "input-checksums.json").read_text())
for relative, expected_hash in hashes.items():
    assert (
        hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected_hash
    ), f"Changed input: {relative}"
print("Mode:", MODE, "Raw counts:", {k: len(v) for k, v in raw.items()})

Mode: pair Raw counts: {'collar': 340, 'survey': 680, 'assay': 8751, 'litho': 1344}


## 2 · QC, coordinate system dan log
**Tindakan:** cek ID, interval, missing dan arah dip. X/Y adalah koordinat; ZCOLLAR adalah RL. Untuk lubang vertikal, elevasi pada kedalaman MD adalah `ZCOLLAR−MD`, bukan `MD`.

**Cek:** raw tidak overlap, seluruh ID ada di collar. Raw Ni missing34, bukan34nilai nol. Survey miring ditolak oleh spesialisasi vertikal notebook ini.

**Diskusi:** jika lubang miring, mengapa garis lurus vertikal dan interpolasi station chord tidak cukup?


In [3]:
collar = raw["collar"].set_index("BHID", verify_integrity=True)
assert all(
    set(df.BHID) <= set(collar.index) for name, df in raw.items() if name != "collar"
)
assert np.isfinite(collar[["XCOLLAR", "YCOLLAR", "ZCOLLAR"]]).all().all()
for name in ("assay", "litho"):
    df = raw[name]
    assert (df.TO > df.FROM).all() and (df.FROM >= 0).all()
    for hole, g in df.groupby("BHID", sort=False):
        ordered = g.sort_values("FROM")
        assert np.all(
            ordered.FROM.to_numpy()[1:] >= ordered.TO.to_numpy()[:-1] - 1e-9
        ), f"Overlap: {name}/{hole}"
# These specific teaching data are vertical. For deviated surveys use the
# GeoSuite measured-survey/minimum-curvature path; do not apply this shortcut.
assert (
    raw["survey"].DIP == 90
).all(), "Notebook vertical specialization cannot process deviated holes"
assert (raw["survey"].AZ == 0).all()
assay = raw["assay"].copy()
assay["X"] = assay.BHID.map(collar.XCOLLAR)
assay["Y"] = assay.BHID.map(collar.YCOLLAR)
assay["Z"] = assay.BHID.map(collar.ZCOLLAR) - (assay.FROM + assay.TO) / 2
print("CRS EPSG:32751; metre; Ni percent. Missing Ni:", assay.NI_PCT.isna().sum())
print("Raw Ni range:", assay.NI_PCT.min(), assay.NI_PCT.max())

CRS EPSG:32751; metre; Ni percent. Missing Ni: 34
Raw Ni range: 0.146 3.657


## 3 · EDA sebelum keputusan geologi
**Tindakan:** baca histogram, missing dan peta observasi. Eksplorasi statistik dapat dimulai sebelum solid final.

**Cek:** angka serta unit sesuai tabel raw. Jangan menggunakan klaster/p-value sebagai bukti tunggal tubuh geologi. Jangan mengubah top-cut agar hasilnya sekadar cocok dengan contoh.


In [4]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(assay.NI_PCT.dropna(), bins=35, color="#087f82")
ax[0].set(xlabel="Ni (%)", ylabel="Raw interval count", title="Raw assay distribution")
scatter = ax[1].scatter(assay.X, assay.Y, c=assay.NI_PCT, s=3, cmap="viridis")
ax[1].set(
    xlabel="Easting (m)", ylabel="Northing (m)", title="Measured midpoint observations"
)
fig.colorbar(scatter, ax=ax[1], label="Ni (%)")
plt.tight_layout()
plt.show()

Actual calculation figure saved at figures/pair-figure-1.png

**Hasil aktual perhitungan mode pair (referensi statis).** Grafik baru akan muncul saat sel dijalankan di kernel.

![Actual pair calculation](figures/pair-figure-1.png)


## 4 · Surface, domain dan solid adalah hal berbeda
**Surface-pair:** SAP memakai Top(SAP) → Bottom(SAP); LIM dapat memakai Top(LIM) → Top(SAP) bila interpretasi log mendukung. **Loft:** ring tertutup pada beberapa section sejajar kemudian dihubungkan.

**Tindakan:** lihat mesh referensi dari perjalanan GeoSuite yang terdokumentasi. Ini adalah keputusan geologi yang diberikan sebagai input, bukan disimpulkan otomatis oleh notebook. Source geometri/hasil historis dipertahankan dalam file referensi; bukan owner acceptance baru.

**Cek:** hitung volume mesh dengan origin lokal. Whole-mesh volume berbeda dari volume blok terisi dan tonase. Density1,6t/m³ hanyalah asumsi latihan.

**SAP berulang1–4m dan6–7m:** gap4–6m bukan SAP. Kode di bawah menolak beberapa perpotongan vertikal yang belum diinterpretasikan, alih-alih membuat envelope palsu.


In [5]:
params = json.loads((REF / "parameters.json").read_text())
solid = json.loads((REF / MODE / "interpreted-solid.json").read_text())
mesh = solid["mesh"]
vertices = np.asarray(mesh["v"], float)
faces = np.asarray(mesh["f"], int)
# Local origin prevents cancellation with million-metre projected coordinates.
local = vertices - vertices[0]
tri = local[faces]
mesh_volume = abs(
    np.einsum("ij,ij->i", tri[:, 0], np.cross(tri[:, 1], tri[:, 2])).sum() / 6
)
reference_volume = params["expected"][MODE]["domains_m3"]["SAP"]
assert abs(mesh_volume - reference_volume) < 1e-5
print("Interpreted SAP whole-mesh volume (not block volume):", mesh_volume, "m³")
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(projection="3d")
ax.plot_trisurf(
    vertices[:, 0] - vertices[0, 0],
    vertices[:, 1] - vertices[0, 1],
    vertices[:, 2],
    triangles=faces,
    alpha=0.35,
    color="#d79d26",
    linewidth=0.05,
)
ax.set(
    xlabel="Easting offset (m)",
    ylabel="Northing offset (m)",
    zlabel="RL (m)",
    title=MODE + " — frozen teaching interpretation",
)
plt.show()

Interpreted SAP whole-mesh volume (not block volume): 15304948.715316301 m³


Actual calculation figure saved at figures/pair-figure-2.png

**Hasil aktual perhitungan mode pair (referensi statis).** Grafik baru akan muncul saat sel dijalankan di kernel.

![Actual pair calculation](figures/pair-figure-2.png)


## 5 · Constrain raw assay dengan solid
**Tindakan:** gunakan ray/triangle untuk memperoleh interval dukungan dalam mesh. Hole di batas lateral tetap dilaporkan tersendiri, bukan diam-diam dianggap inside.

**Cek pair:**302hole interior dan38hole boundary.4.122m support interior dibanding468m boundary yang tidak otomatis assigned. Semua340rawhole tetap disimpan.

**Diskusi:** boundary policy harus ditetapkan sebelum menjumlah tonase atau menyebut data hilang.


In [6]:
# Ray/triangle intersection is independent NumPy math. This training mesh has
# one continuous vertical intersection per interior hole. Never replace
# multiple disjoint intervals by a first-top/last-bottom envelope.
anchor = vertices[0]
t = local[faces]
a = t[:, 0, :2]
u = t[:, 1, :2] - a
v = t[:, 2, :2] - a
det = u[:, 0] * v[:, 1] - u[:, 1] * v[:, 0]
horizontal = np.abs(det) > 1e-10
side = t[~horizontal, :, :2]
ranges = {}
boundary = {}
for hole, c in collar.iterrows():
    point = np.array([c.XCOLLAR, c.YCOLLAR]) - anchor[:2]
    q = point - a
    w1 = np.zeros(len(t))
    w2 = w1.copy()
    w1[horizontal] = (
        q[horizontal, 0] * v[horizontal, 1] - q[horizontal, 1] * v[horizontal, 0]
    ) / det[horizontal]
    w2[horizontal] = (
        u[horizontal, 0] * q[horizontal, 1] - u[horizontal, 1] * q[horizontal, 0]
    ) / det[horizontal]
    hits = horizontal & (w1 >= -1e-10) & (w2 >= -1e-10) & (w1 + w2 <= 1 + 1e-10)
    heights = (
        t[hits, 0, 2]
        + w1[hits] * (t[hits, 1, 2] - t[hits, 0, 2])
        + w2[hits] * (t[hits, 2, 2] - t[hits, 0, 2])
        + anchor[2]
    )
    heights = np.asarray(sorted(heights))
    unique = []
    for z in heights:
        if not unique or abs(z - unique[-1]) > 1e-7:
            unique.append(z)
    if len(unique) < 2:
        continue
    assert (
        len(unique) == 2
    ), f"Multiple intersections need explicit separate bodies: {hole}"
    on_boundary = False
    for triangle in side:
        pairs = [(triangle[i], triangle[j]) for i, j in ((0, 1), (1, 2), (2, 0))]
        start, end = max(pairs, key=lambda pair: np.linalg.norm(pair[1] - pair[0]))
        edge = end - start
        length2 = edge @ edge
        if length2:
            along = (point - start) @ edge / length2
            if (
                -1e-12 <= along <= 1 + 1e-12
                and np.linalg.norm(point - start - along * edge) <= 1e-8
            ):
                on_boundary = True
                break
    target = boundary if on_boundary else ranges
    target[hole] = (float(c.ZCOLLAR - max(unique)), float(c.ZCOLLAR - min(unique)))
print(
    "Interior holes:", len(ranges), "Boundary holes excluded explicitly:", len(boundary)
)
if MODE == "pair":
    assert len(ranges) == 302 and len(boundary) == 38

Interior holes: 302 Boundary holes excluded explicitly: 38


## 6 · Komposit2m dengan bobot panjang
Rumus: `grade = Σ(length × grade) / Σ(informed length)`; missing tidak menjadi nol. Tail tetap ditampilkan. Input kadar berasal dari **raw assay**, bukan menyalin kadar komposit referensi.

**Cek:** pair2.139composite, loft124composite. Perbandingan ekspor mempertimbangkan presisi depth6desimal dan grade10desimal; bukan toleransi persen universal. Nilai dan dukungan ditulis ke hasil-python/.

**Latihan:** periksa satu komposit secara manual dan bandingkan tail, panjang fisik serta panjang yang memiliki Ni.


In [7]:
rows = []
support_total = 0.0
support_grade_total = 0.0
for hole, (start, end) in ranges.items():
    source = assay[assay.BHID == hole]
    for lo in np.arange(start, end - 1e-8, 2.0):
        hi = min(lo + 2, end)
        lengths = np.maximum(
            0, np.minimum(source.TO, hi) - np.maximum(source.FROM, lo)
        ).to_numpy()
        physical = float(lengths.sum())
        if physical <= 1e-8:
            continue
        informed = np.isfinite(source.NI_PCT.to_numpy())
        informed_support = float(lengths[informed].sum())
        weighted = float(lengths[informed] @ source.NI_PCT.to_numpy()[informed])
        grade = weighted / informed_support if informed_support else np.nan
        c = collar.loc[hole]
        rows.append(
            {
                "hole_id": hole,
                "from_m": lo,
                "to_m": hi,
                "informed_length": physical,
                "ni_support_m": informed_support,
                "ni_pct": grade,
                "midx": c.XCOLLAR,
                "midy": c.YCOLLAR,
                "midz": c.ZCOLLAR - (lo + hi) / 2,
            }
        )
        support_total += physical
        support_grade_total += weighted
composites = pd.DataFrame(rows)
ref_composite = pd.read_csv(REF / MODE / "composites-2m.csv")
assert (
    len(composites) == len(ref_composite) == params["expected"][MODE]["composites_2m"]
)
# Key by hole and deterministic interval order; exports round depths to six
# decimals, so exact binary equality of printed coordinates is not appropriate.
left = composites.sort_values(["hole_id", "from_m"]).reset_index(drop=True)
right = ref_composite.sort_values(["hole_id", "from_m"]).reset_index(drop=True)
assert (left.hole_id == right.hole_id).all()
assert np.max(np.abs(left.from_m - right.from_m)) <= 5e-7 + 1e-9
assert np.max(np.abs(left.to_m - right.to_m)) <= 5e-7 + 1e-9
assert np.array_equal(np.isnan(left.ni_pct), np.isnan(right.ni_pct))
# Interval export precision can shift both support endpoints. Conservative
# length-weighted mean error bound: 4*epsilon*max|grade|/(informed_support-2eps).
eps = 5e-7
max_grade = assay.NI_PCT.abs().max()
tolerance = 4 * eps * max_grade / np.maximum(left.ni_support_m - 2 * eps, 1e-9) + 5e-11
finite = np.isfinite(left.ni_pct)
grade_diff = np.abs(left.ni_pct - right.ni_pct)
assert (grade_diff[finite] <= tolerance[finite]).all()
print(
    "Composites:",
    len(composites),
    "Physical support:",
    support_total,
    "m; missing composite Ni:",
    composites.ni_pct.isna().sum(),
)
print(
    "Max composite grade difference vs GeoSuite export:",
    grade_diff.max(),
    "percentage points",
)
composites.to_csv(RESULTS / f"{MODE}-composites-python.csv", index=False)

Composites: 2139 Physical support: 4122.0 m; missing composite Ni: 1
Max composite grade difference vs GeoSuite export: 3.064215547965432e-14 percentage points


## 7 · Pasangan variogram dan model
**Tindakan:** hitung pasangan eksperimen dan overlay model spherical yang dinyatakan pada parameter GeoSuite. Grafik Python ini **omnidirectional**, bukan klaim meniru semua pilihan directional/fit UI. Arah, lag, toleransi dan jumlah pasangan harus diperiksa ketika menginterpretasi model.

Geometri komposit presisi penuh menjadi input yang eksplisit dan dibandingkan terhadap koordinat analitik vertikal; beda round-trip harus <1µm. Jangan memakai angka koordinat yang dibulatkan dari screenshot/CSV display untuk benchmark dekat tie tetangga.

**Cek:** nugget/sill/range tercatat; covariance model menggunakan jarak fisik, ellipsoid search mengatur populasi tetangga. Keduanya bukan parameter yang sama.


In [8]:
samples = composites.dropna(subset=["ni_pct"]).reset_index(drop=True)
geometric_input = json.loads((REF / MODE / "geometry-full-precision.json").read_text())
assert [s["hole_id"] for s in geometric_input["samples"]] == samples.hole_id.tolist()
xyz_computed = samples[["midx", "midy", "midz"]].to_numpy()
xyz = np.asarray([s["xyz"] for s in geometric_input["samples"]], float)
assert np.max(np.abs(xyz - xyz_computed)) < 1e-6
# Geometry is an explicit shared geological input, now with full floating-point
# precision. Use raw-recomputed grades, never reference estimated block grades.
grades = samples.ni_pct.to_numpy()
print("Max geometric round-trip difference (m):", np.max(np.abs(xyz - xyz_computed)))
distance = pdist(xyz)
semivariance = 0.5 * pdist(grades[:, None], metric="sqeuclidean")
edges = np.arange(0, 501, 50)
empirical = []
for low, high in zip(edges[:-1], edges[1:]):
    selected = (distance > low) & (distance <= high)
    empirical.append(
        {
            "lag_m": (low + high) / 2,
            "pairs": int(selected.sum()),
            "gamma": float(semivariance[selected].mean()) if selected.any() else np.nan,
        }
    )
variogram = pd.DataFrame(empirical)
model = params["models"][MODE]


def gamma(h):
    d = np.asarray(h, float)
    r = d / model["range"]
    partial = model["sill"] - model["nugget"]
    return np.where(
        d <= 0,
        0.0,
        model["nugget"] + partial * np.where(r >= 1, 1.0, 1.5 * r - 0.5 * r**3),
    )


fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(variogram.lag_m, variogram.gamma, label="Omnidirectional pairs (Python)")
h = np.linspace(0, 600, 250)
ax.plot(h, gamma(h), label="Supplied GeoSuite spherical model")
ax.set(
    xlabel="Distance (m)",
    ylabel="Semivariance (%²)",
    title="Pair counts, direction and support must be reviewed",
)
ax.legend()
plt.show()
print(variogram.to_string(index=False))

Max geometric round-trip difference (m): 5.684341886080802e-14
 lag_m  pairs    gamma
  25.0  31259 0.156668
  75.0  75904 0.172872
 125.0  94140 0.183754
 175.0 103692 0.198220
 225.0 118178 0.211067
 275.0 137419 0.227400
 325.0 149332 0.246508
 375.0 152174 0.258613
 425.0 148126 0.255169
 475.0 142322 0.251578


Actual calculation figure saved at figures/pair-figure-3.png

**Hasil aktual perhitungan mode pair (referensi statis).** Grafik baru akan muncul saat sel dijalankan di kernel.

![Actual pair calculation](figures/pair-figure-3.png)


## 8 · Block support dan estimasi NN/IDW/OK
**Tindakan:** gunakan support blok terpotong dari interpretasi GeoSuite sebagai input yang dinyatakan jelas. Notebook tidak mengklaim menghitung ulang seluruh clipping volume3D secara independen. Kadar blok referensi hanya dipakai pada sel perbandingan, tidak sebagai input estimasi.

Search baseline:250/250/50m, azimuth0°, dip0°, minimum2 dan maksimum12tetangga, tanpa octant/hole-cap/second-pass. NN memilih satu observasi; IDW memakai jarak fisik berpangkat2; OK menyelesaikan KKT dengan bobot berjumlah1. Stabilizer diagonal1e-9 mengikuti konvensi benchmark yang dinyatakan, bukan menutupi singularitas.

**Cek:** pair2.237blok terestimasi; loft56. Unknown/-999 pada ekspor bukan kadar negatif yang boleh dirata-ratakan. Lingkungan/model lanjutan di luar konfigurasi ini memerlukan verifikasi tambahan.


In [9]:
# Supplied clipped block support is a disclosed geological input from GeoSuite,
# NOT independently recreated by this notebook. No grade is read as an input.
block_reference = pd.read_csv(REF / MODE / "block-model-reference.csv")
blocks = (
    block_reference[block_reference.SUPPORT_VOLUME_M3 > 0].copy().reset_index(drop=True)
)
search = params["search"]
radius = np.array([search["rSemi"], search["rMaj"], search["rMin"]])
assert (
    search["azDeg"] == 0
    and search["dipDeg"] == 0
    and search["minNeighbors"] == 2
    and search["maxNeighbors"] == 12
)
assert (
    search["maxPerHole"] == 0 and not search["useOctant"] and not search["secondPass"]
)


def estimate(target, exclude=None):
    delta = xyz - np.asarray(target)
    ellipsoid = np.linalg.norm(delta / radius, axis=1)
    valid = ellipsoid <= 1 + 1e-12
    if exclude is not None:
        valid[exclude] = False
    indices = np.flatnonzero(valid)
    # Select top-k by strict normalized distance. The 1e-10 tie rule belongs
    # only to NN within the selected population; it must not change IDW/OK inputs.
    indices = indices[np.argsort(ellipsoid[indices], kind="stable")[:12]]
    if len(indices) < 2:
        return [np.nan, np.nan, np.nan, len(indices), np.nan]
    points = xyz[indices]
    values = grades[indices]
    dist = np.linalg.norm(points - target, axis=1)
    weights = np.where(dist <= 1e-12, 1e9, 1 / np.maximum(dist, 1e-12) ** 2)
    idw = float(weights @ values / weights.sum())
    n = len(indices)
    A = np.ones((n + 1, n + 1))
    A[-1, -1] = 0
    A[:n, :n] = gamma(cdist(points, points))
    np.fill_diagonal(A[:n, :n], 1e-9)
    rhs = np.r_[gamma(dist), 1.0]
    solution = solve(A, rhs, assume_a="sym")
    ok = max(0.0, float(solution[:n] @ values))
    assert abs(solution[:n].sum() - 1) < 1e-9
    solver_tolerance = max(
        2e-11,
        100
        * np.finfo(float).eps
        * np.linalg.cond(A)
        * max(1, np.abs(solution).max(), np.abs(values).max()),
    )
    nearest = min(
        (int(i) for i in indices if abs(ellipsoid[i] - ellipsoid[indices[0]]) <= 1e-10)
    )
    return [float(grades[nearest]), idw, ok, n, solver_tolerance]


targets = np.asarray(geometric_input["block_centres"], float)
assert targets.shape == (len(blocks), 3)
# Display CSV has three decimal places; it must not replace full-precision
# centre inputs near a search-neighbour tie. This is input equivalence, not
# parameter tuning to match a result.
assert np.max(np.abs(targets - blocks[["X", "Y", "Z"]].to_numpy())) <= 0.0005 + 1e-9
calculated = np.asarray([estimate(target) for target in targets])
for i, name in enumerate(
    ["NN_python", "IDW_python", "OK_python", "neighbors_python", "solver_tolerance"]
):
    blocks[name] = calculated[:, i]
assert blocks.OK_python.notna().all()
print("Estimated supported blocks:", len(blocks))

Estimated supported blocks: 2237


## 9 · Bandingkan hasil secara kuantitatif
**Tindakan:** baca max difference, RMSE dan bias; periksa scatterplot, bukan hanya angka total.

Batas perbandingan diturunkan dari penyimpanan grade **Float32**, pembulatan ekspor **6desimal**, serta condition-aware floating-point solver. Tidak ada toleransi persentase generik. Jika assertion gagal, hentikan dan periksa input/config; jangan mengganti parameter supaya hijau.

**Batas validasi:** geometri dan block support diberikan oleh GeoSuite. Agreement di sini memeriksa kalkulasi pada input yang setara, bukan validasi geologi atau review profesional independen. Benchmark pihak ketiga PyKrige yang pernah dilakukan adalah bukti terpisah; notebook ini tidak menjalankannya.


In [10]:
comparison = []
for name in ["NN", "IDW", "OK"]:
    predicted = blocks[name + "_python"].to_numpy()
    reference = blocks[name].to_numpy()
    error = predicted - reference
    # GeoSuite stores grades as Float32 and the CSV prints six decimal places.
    # Bound both representations plus a condition-aware double-precision solve.
    limit = 5e-7 + np.abs(np.spacing(reference.astype(np.float32))) / 2
    if name == "OK":
        limit = limit + blocks.solver_tolerance.to_numpy()
    else:
        limit = limit + 3e-12
    assert np.all(
        np.abs(error) <= limit
    ), f"{name} numerical discrepancy beyond stated precision bound"
    comparison.append(
        {
            "method": name,
            "blocks": len(error),
            "max_abs_percentage_points": float(np.max(np.abs(error))),
            "RMSE_percentage_points": float(np.sqrt(np.mean(error**2))),
            "bias_percentage_points": float(error.mean()),
            "precision_check": "PASS",
            "tolerance_basis": "Float32 half-ULP + six-decimal export half-unit + solver bound",
        }
    )
comparison = pd.DataFrame(comparison)
print(comparison.to_string(index=False))
# No universal percentage tolerance: the bound above follows the actual native
# storage/export precision. Given mesh, coordinates and clipped block support
# are shared inputs; this is not an independent geological geometry validation.
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(blocks.OK, blocks.OK_python, s=5)
lo = min(blocks.OK.min(), blocks.OK_python.min())
hi = max(blocks.OK.max(), blocks.OK_python.max())
ax.plot([lo, hi], [lo, hi], color="black")
ax.set(
    xlabel="GeoSuite OK export Ni (%)",
    ylabel="Python OK Ni (%)",
    title="Actual block comparison, same supported grid",
)
plt.show()

method  blocks  max_abs_percentage_points  RMSE_percentage_points  bias_percentage_points precision_check                                                tolerance_basis
    NN    2237               2.198242e-14            2.273682e-15           -3.445811e-16            PASS Float32 half-ULP + six-decimal export half-unit + solver bound
   IDW    2237               5.810293e-07            2.918971e-07            8.299390e-09            PASS Float32 half-ULP + six-decimal export half-unit + solver bound
    OK    2237               6.085750e-07            2.882833e-07            8.430604e-09            PASS Float32 half-ULP + six-decimal export half-unit + solver bound


Actual calculation figure saved at figures/pair-figure-4.png

**Hasil aktual perhitungan mode pair (referensi statis).** Grafik baru akan muncul saat sel dijalankan di kernel.

![Actual pair calculation](figures/pair-figure-4.png)


## 10 · Density, cutoff dan grade–tonnage
Rumus: `tonnes = supported volume × density`; `contained Ni = tonnes × Ni% /100`. Mean grade harus dibobot tonase.

**Tindakan:** lihat perubahan cutoff0–3%Ni dan cocokkan jumlah massa pada cutoff0 dengan scope blok yang sama. Mesh seluruh domain, volume terisi, tonase yang diestimasi dan unestimated support harus dibedakan.

**Diskusi:** bagaimana bila density berbeda per litologi? Mengapa kurva ini belum membuktikan RPEEE, klasifikasi atau cadangan?


In [11]:
DENSITY = params["grid"]["density_t_m3"]
tonnes = blocks.SUPPORT_VOLUME_M3.to_numpy() * DENSITY
gt = []
for cutoff in params["grade_tonnage"]["cutoffs_percent"]:
    selected = blocks.OK_python.to_numpy() >= cutoff
    mass = float(tonnes[selected].sum())
    metal = float(tonnes[selected] @ blocks.OK_python.to_numpy()[selected] / 100)
    gt.append(
        {
            "cutoff_Ni_pct": cutoff,
            "tonnes": mass,
            "mean_Ni_pct": 100 * metal / mass if mass else np.nan,
            "contained_Ni_t": metal,
        }
    )
gt = pd.DataFrame(gt)
print(gt.to_string(index=False))
assert np.all(np.diff(gt.tonnes) <= 1e-6)
assert abs(float(tonnes.sum()) - params["expected"][MODE]["estimated"]["tonnes"]) < 1e-4
assert float(blocks.SUPPORT_VOLUME_M3.sum()) <= mesh_volume + 1e-5
# Reconcile aggregate grade and metal against the native Float32 snapshot,
# using the tonnage-weighted representation error, not an arbitrary percent.
native_half_ulp = np.abs(np.spacing(blocks.OK.to_numpy().astype(np.float32))) / 2
mean_tolerance = float(
    tonnes @ native_half_ulp / tonnes.sum() + blocks.solver_tolerance.max() + 1e-10
)
metal_tolerance = float(
    tonnes @ native_half_ulp / 100
    + tonnes.sum() * (blocks.solver_tolerance.max() + 1e-10) / 100
    + 1e-4 * grades.max() / 100
)
ref_estimate = params["expected"][MODE]["estimated"]
assert round(float(tonnes.sum()), 2) == round(
    ref_estimate["tonnes"], 2
), "Final displayed tonnes must match GeoSuite"
assert abs(gt.iloc[0].mean_Ni_pct - ref_estimate["meanGrade"]) <= mean_tolerance
assert abs(gt.iloc[0].contained_Ni_t - ref_estimate["metalTonnes"]) <= metal_tolerance
print(
    "Aggregate reconciliation PASS; mean-grade tolerance:",
    mean_tolerance,
    "percentage points; contained-metal tolerance:",
    metal_tolerance,
    "t",
)
final_check = pd.DataFrame(
    [
        {
            "metric": "Supported volume (m3)",
            "Python": float(blocks.SUPPORT_VOLUME_M3.sum()),
            "GeoSuite": ref_estimate["volume"],
            "absolute_difference": abs(
                float(blocks.SUPPORT_VOLUME_M3.sum()) - ref_estimate["volume"]
            ),
            "tolerance": 1e-5,
        },
        {
            "metric": "Final tonnage (t)",
            "Python": float(tonnes.sum()),
            "GeoSuite": ref_estimate["tonnes"],
            "absolute_difference": abs(float(tonnes.sum()) - ref_estimate["tonnes"]),
            "tolerance": 1e-4,
        },
        {
            "metric": "Mean Ni (%)",
            "Python": float(gt.iloc[0].mean_Ni_pct),
            "GeoSuite": ref_estimate["meanGrade"],
            "absolute_difference": abs(
                float(gt.iloc[0].mean_Ni_pct) - ref_estimate["meanGrade"]
            ),
            "tolerance": mean_tolerance,
        },
        {
            "metric": "Contained Ni (t)",
            "Python": float(gt.iloc[0].contained_Ni_t),
            "GeoSuite": ref_estimate["metalTonnes"],
            "absolute_difference": abs(
                float(gt.iloc[0].contained_Ni_t) - ref_estimate["metalTonnes"]
            ),
            "tolerance": metal_tolerance,
        },
    ]
)
final_check["result"] = np.where(
    final_check.absolute_difference <= final_check.tolerance, "PASS", "FAIL"
)
assert (final_check.result == "PASS").all()
print("FINAL UI / NOTEBOOK CHECK, SAME DOMAIN / SUPPORT / DENSITY / CUTOFF:")
print(final_check.to_string(index=False))
print(
    f"Final tonnage: {tonnes.sum():,.2f} t; mode={MODE}; cutoff=0% Ni; density={DENSITY} t/m3"
)
final_check.to_csv(RESULTS / f"{MODE}-final-ui-comparison.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gt.cutoff_Ni_pct, gt.tonnes / 1e6, marker="o")
ax.set(
    xlabel="Cutoff Ni (%)",
    ylabel="Supported tonnes (Mt)",
    title="Training grade–tonnage: uniform density assumption",
)
plt.show()

 cutoff_Ni_pct       tonnes  mean_Ni_pct  contained_Ni_t
           0.0 2.129833e+07     1.700600   362199.366098
           0.5 2.129833e+07     1.700600   362199.366098
           1.0 2.128160e+07     1.701153   362032.611497
           1.5 1.565578e+07     1.830722   286613.776995
           2.0 3.273464e+06     2.188679    71645.621792
           2.5 2.051333e+05     2.634977     5405.217241
           3.0 0.000000e+00          NaN        0.000000
Aggregate reconciliation PASS; mean-grade tolerance: 6.886222283436816e-08 percentage points; contained-metal tolerance: 0.01466976766271982 t
FINAL UI / NOTEBOOK CHECK, SAME DOMAIN / SUPPORT / DENSITY / CUTOFF:
               metric       Python     GeoSuite  absolute_difference    tolerance result
Supported volume (m3) 1.331146e+07 1.331146e+07         1.303852e-08 1.000000e-05   PASS
    Final tonnage (t) 2.129833e+07 2.129833e+07         3.352761e-08 1.000000e-04   PASS
          Mean Ni (%) 1.700600e+00 1.700600e+00         8.663625e

Actual calculation figure saved at figures/pair-figure-5.png

**Hasil aktual perhitungan mode pair (referensi statis).** Grafik baru akan muncul saat sel dijalankan di kernel.

![Actual pair calculation](figures/pair-figure-5.png)


## 11 · Evaluasi, export dan pertanyaan akhir
**Tindakan:** lihat contoh leave-one-observation-out maksimal30observasi dan simpan CSV/JSON hasil. Ini bukan leave-one-hole-out dan **bukan** pengganti konfigurasi CV yang dipilih di GeoSuite.

**Cek:** hasil berulang deterministik dengan input/parameter yang sama. Perubahan MODE membutuhkan Run All. Folder hasil-python/ milik peserta; jangan menimpa raw dataset/.

**Kesimpulan yang boleh disampaikan:** kalkulasi Python pada dataset latihan sudah diuji dengan scope dan batas yang dinyatakan. Bukan bukti semua39layar accepted, bukan uji geolog eksternal, bukan compliant resource statement, dan bukan approval production.


In [12]:
# Leave-one-observation-out is teaching diagnostics, NOT independent drillhole
# validation and not a substitute for the GeoSuite chosen CV configuration.
selection = np.arange(min(30, len(samples)))
cv = np.asarray([estimate(xyz[i], exclude=i)[2] for i in selection])
mask = np.isfinite(cv)
print(
    "Teaching LOO observations:",
    int(mask.sum()),
    "RMSE:",
    float(np.sqrt(np.mean((cv[mask] - grades[selection][mask]) ** 2))),
)
comparison.to_csv(RESULTS / f"{MODE}-comparison.csv", index=False)
gt.to_csv(RESULTS / f"{MODE}-grade-tonnage.csv", index=False)
blocks.to_csv(RESULTS / f"{MODE}-block-estimates-python.csv", index=False)
report = {
    "mode": MODE,
    "raw_counts": expected_counts,
    "mesh_volume_m3": mesh_volume,
    "composites": len(composites),
    "composite_physical_support_m": support_total,
    "estimated_blocks": len(blocks),
    "supported_volume_m3": float(blocks.SUPPORT_VOLUME_M3.sum()),
    "tonnes": float(tonnes.sum()),
    "comparison": comparison.to_dict("records"),
    "limits": [
        "synthetic educational data",
        "vertical survey specialization",
        "frozen interpreted mesh and clipped block support provided",
        "not a geological or professional independent sign-off",
        "not SNI/KCMI/JORC compliance",
        "no raw grades modified",
    ],
}
(RESULTS / f"{MODE}-run-summary.json").write_text(json.dumps(report, indent=2) + "\n")
print(json.dumps(report, indent=2))

Teaching LOO observations: 30 RMSE: 0.41996620302914633
{
  "mode": "pair",
  "raw_counts": {
    "collar": 340,
    "survey": 680,
    "assay": 8751,
    "litho": 1344
  },
  "mesh_volume_m3": 15304948.715316301,
  "composites": 2139,
  "composite_physical_support_m": 4122.0,
  "estimated_blocks": 2237,
  "supported_volume_m3": 13311455.976011543,
  "tonnes": 21298329.56161847,
  "comparison": [
    {
      "method": "NN",
      "blocks": 2237,
      "max_abs_percentage_points": 2.19824158875781e-14,
      "RMSE_percentage_points": 2.273682465159367e-15,
      "bias_percentage_points": -3.44581066605832e-16,
      "precision_check": "PASS",
      "tolerance_basis": "Float32 half-ULP + six-decimal export half-unit + solver bound"
    },
    {
      "method": "IDW",
      "blocks": 2237,
      "max_abs_percentage_points": 5.810292695862529e-07,
      "RMSE_percentage_points": 2.9189708497326566e-07,
      "bias_percentage_points": 8.299390418588602e-09,
      "precision_check": "PASS",


## Referensi dan tindak lanjut
- Data: Orebit Synthetic Drillhole Datasets, source6b93ac48/02-nikel-laterit; subset340whole-hole, CC BY4.0.
- Konfigurasi dan provenance: output/parameters.json, interpreted-solid.json dan geometry-full-precision.json; source historis dipertahankan.
- CIM2019 MRMR Best Practice: EDA6.2, compositing6.4, density6.5, continuity6.8, model validation6.10; dipakai sebagai konteks metodologi, bukan sertifikasi software.
- JORC2012 Table1 Section3 dan SNI4726:2019 memerlukan konteks/kompetensi pelaporan; tonase dan kadar dari notebook tidak otomatis memenuhi ketentuan tersebut.
- Untuk bug, catat MODE, checksum input, parameter, sel gagal dan pesan. Jangan kirim credential atau data pelanggan.

**Latihan mandiri:** jelaskan pilihan geologi, support yang dikeluarkan, pengaruh komposit, pilihan model/search, dan mengapa total surface-pair tidak boleh dibandingkan langsung dengan loft yang footprint-nya berbeda.
